# 1. Qwen2.5-1.5B-Instruct

**모델:** `Qwen/Qwen2.5-1.5B-Instruct`  
**모델 카드:** https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct  
**파라미터:** 1.54B (약 1.5B)  
**만든 곳:** Qwen Team / Alibaba Cloud

## 모델 설명

Qwen2.5-1.5B-Instruct는 Alibaba Cloud의 Qwen Team이 만든 Qwen2.5 계열 instruction-tuned 모델입니다.  
- **Instruct 모델**: 질문을 입력하면 바로 답변 생성 (대화 바로 가능)
- 한국어, 영어, 중국어 등 다양한 언어 입력 처리 가능
- 1.5B급 소형 모델로 Colab, 로컬 GPU, 실습 환경에서 비교적 가볍게 실행 가능
- `trust_remote_code` 없이 표준 transformers로 바로 사용 가능
- `tokenizer.apply_chat_template()`을 사용해 Qwen 대화 형식으로 입력 구성 가능

In [ ]:
# https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct

In [ ]:
%pip install transformers accelerate ipywidgets hf_xet
# 라이브러리 설치 transformers accelerate ipywidgets hf_xet

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

In [ ]:
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

print(generated_ids)
print('='*100)

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

In [ ]:
prompt = "hello"
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]

print(messages)
print("="*100)

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print(text)
print("="*100)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
print(model_inputs)

# 2. Kanana-nano-2.1b-instruct

**모델:** `kakaocorp/kanana-nano-2.1b-instruct`  
**모델 카드:** https://huggingface.co/kakaocorp/kanana-nano-2.1b-instruct  
**파라미터:** 2.1B  
**만든 곳:** 카카오 (Kakao Corp.)

## 모델 설명

Kanana Nano는 카카오가 만든 한국어/영어 이중언어 instruction-tuned 모델입니다.  
- **Instruct 모델**: 질문을 입력하면 바로 답변 생성 (대화 바로 가능)
- 한국어 성능이 우수하며, 2.1B 파라미터로 소형 기기에서도 실행 가능
- `trust_remote_code` 없이 표준 transformers로 바로 사용 가능
- CPU에서도 실행 가능 (단, 응답 생성에 수십 초 소요)

In [ ]:
# https://huggingface.co/kakaocorp/kanana-nano-2.1b-instruct
# 모델 에러 해결 - 주로 trasnformers의 버전 문제일 가능성이 높다.
# uv remove transformers
# uv add transformers==4.45.0

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
from huggingface_hub import login
import os

login(token=os.getenv("HUGGINGFACE_TOKEN"))

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

model_name = 'kakaocorp/kanana-nano-2.1b-instruct'

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,  # CPU는 float32 사용
)

In [ ]:
# 질문 바로 입력 → 답변 생성
messages = [
    {"role": "system", "content": "You are a helpful AI assistant developed by Kakao."},
    {"role": "user", "content": "한국의 수도는 어디야? 간단히 알려줘."}
]

model_inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

In [ ]:
with torch.no_grad():
    output = model.generate(
        **model_inputs,
        max_new_tokens=100,
        do_sample=False,
    )

# 입력 부분 제외하고 생성된 답변만 출력
response = tokenizer.decode(output[0][model_inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("[질문] 한국의 수도는 어디야? 간단히 알려줘.")
print(f"[답변] {response}")

# 3. Gemma-3-1b-it

**모델:** `google/gemma-3-1b-it`  
**모델 카드:** https://huggingface.co/google/gemma-3-1b-it  
**파라미터:** 1B  
**만든 곳:** Google DeepMind

## 모델 설명

Gemma 3 1B IT는 Google DeepMind가 만든 Gemma 3 계열의 instruction-tuned 텍스트 생성 모델입니다.  
- **Instruct 모델**: 질문을 입력하면 바로 답변 생성 (대화 바로 가능)
- 1B 파라미터의 소형 모델로 실습, 빠른 테스트, 제한된 GPU 환경에서 사용하기 적합
- Gemma 3 계열 모델로 한국어를 포함한 여러 언어 입력 처리 가능
- `transformers` 4.50.0 이상에서 `pipeline` 또는 `Gemma3ForCausalLM`로 사용 가능
- 모델 사용 전 Hugging Face에서 Gemma 라이선스/사용 조건 동의가 필요할 수 있음

In [ ]:
# https://huggingface.co/google/gemma-3-1b-it
# 모델 불러오기 에러 해결 과정
# uv remove transformers
# uv add transformers==4.50.0
# uv add bitsandbytes

In [ ]:
from huggingface_hub import login
import os

login(token=os.getenv("HUGGINGFACE_TOKEN"))

In [ ]:
from transformers import pipeline
import torch

pipe = pipeline(
    "text-generation",
    model="google/gemma-3-1b-it",
    # device="cuda",
    # dtype=torch.bfloat16
)

messages = [
    [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are a helpful assistant."},]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": "Write a poem on Hugging Face, the company"},]
        },
    ],
]

output = pipe(messages, max_new_tokens=50)


In [ ]:
print(output)

In [ ]:
for out in output:
    print(out)

In [ ]:
from transformers import AutoTokenizer, BitsAndBytesConfig, Gemma3ForCausalLM
import torch

model_id = "google/gemma-3-1b-it"

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = Gemma3ForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
).eval()

tokenizer = AutoTokenizer.from_pretrained(model_id)


In [ ]:
messages = [
    [
        {
            "role": "system",
            "content": [{"type": "text", "text": "You are a helpful assistant."},]
        },
        {
            "role": "user",
            "content": [{"type": "text", "text": "Write a poem on Hugging Face, the company"},]
        },
    ],
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device).to(torch.bfloat16)


In [ ]:
with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=64)

outputs = tokenizer.batch_decode(outputs)


In [ ]:
print(outputs)

In [ ]:
for out in outputs:
    print(out)

# 파인튜닝 데이터셋은 어떻게 생겼을까?

In [ ]:
# messages = [
#     [
#         {
#             "role": "system",
#             "content": [{"type": "text", "text": "You are a helpful assistant."},]
#         },
#         {
#             "role": "user",
#             "content": [{"type": "text", "text": "Write a poem on Hugging Face, the company"},]
#         },
#     ],
# ]

## 1) Dataset 이해하기

In [ ]:
#uv add datasets

In [ ]:
# beomi/KoAlpaca-v1.1a
%pip install datasets
from datasets import load_dataset

dataset = load_dataset('beomi/KoAlpaca-v1.1a')
dataset

In [ ]:
dataset['train']

In [ ]:
dataset['train'][10]

## 2) datasets 기능으로 데이터셋 만들기

In [ ]:
sampledata = dataset['train'].select(range(10))
sampledata

In [ ]:
sampledata[0]

In [ ]:
# data가 들어왔을 때 messages = [{"role":, "content":},{"role":, "content":}] 형태로 만드는 함수 만들기
def make_prompt(data):
    return {
        'result': [
            {'role': 'user', 'content': data['instruction']},
            {'role': 'user', 'content': data['output']}
        ]
    }

In [ ]:
make_prompt(sampledata[0])

In [ ]:
test = sampledata.map(make_prompt)
test